# **Aula prática:** Problema de Transbordo com Custos Fixos e Variáveis

Já trabalhamos com um nutricionista, uma empresa de serviços terceirizados e no suporte à tomada de decisão da CBDA. Hoje a AMBEV está precisando dos nossos serviços, mas este mesmo problema é também útil para empresas como fazendas de gado, plataformas de comércio eletrônico (Amazon, Mercado Livro, SHEIN, etc.) e até mesmo o MC Donalds.




In [ ]:
#ao executar este código, instalamos o suporte ao AMPL no notebook e no python
!pip install -q amplpy
from amplpy import tools
ampl = tools.ampl_notebook(
    modules=["highs", "coin"], # pick from over 20 modules including most commercial and open-source solvers
    license_uuid="bcc3d88a-8b8c-4c93-8f21-c2bedd3fc48f") # your license UUID

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 27.8 MB/s eta 0:00:00
Licensed to AMPL Community Edition License for <santi.everton@gmail.com>.


## Descrição do Problema

Você foi contratado por uma empresa do ramo de bebidas para resolver o problema que é descrito a seguir, tendo em vista a proximidade do feriado de carnaval.

A empresa possui $m$ fábricas, $n$ centros de distribuição e $p$ clientes que devem ser atendidos. As fábricas, centros de distribuição e clientes estão localizados em diversos locais pelo Brasil. Portanto, o custo de transporte de produtos entre fábricas e centros, bem como centros e clientes, pode variar de acordo com a distância, pedágios ou até mesmo o modal que é utilizado para fazer este transporte.

Considerando um produto específico, a empresa estima que no feriado de carnaval cada cliente $k~(k=1, 2, ..., p)$ terá uma demanda específica de $d_k$ unidades por este produto.

A empresa estima também que produzir uma unidade do produto na fábrica $i$ e transportá-la para o centro de distribuição $j$ tem um custo $c_{ij}$, para todo $i=1, 2, ..., m$ e para todo $j=1, 2, ..., n$. De maneira similar, armazenar e transportar uma unidade do produto do centro de distribuição $j$ para o cliente $k$ apresenta um custo $t_{jk}$, para todo $j=1, 2, ..., n$ e para todo $k=1,2, ...,p$.

Em relação ao processo de **produção**, sabe-se que uma fábrica $i$ tem um custo fixo de produção $f_i$. Isto é, se ela é usada para produzir uma ou mais unidades do produto, há um custo a ser pago, relacionado à configuração da linha de produção, funcionários, dentre outros recursos operacionais que serão disponibilizados. Cada uma destas fábricas possui também uma capacidade de produção, representada por $g_i$.

Em relação ao processo de **distruibuição**, sabe-se que um centro $j$ tem um custo fixo de operação $h_j$. Isto é, se ele é usado armazenar e transportar uma ou mais unidades do produto, há um custo a ser pago, relacionado à configuração do centro, como alocação dos funcionários, destinação do espaço, energia elétrica, dentre outros recursos. Cada um destes centros possui também uma capacidade de armazenando e processamento, representado por $q_j$.

## Formulação

Para modelar o problema, podemos utilizar as seguintes **variáveis de decisão**:

* $xx_i$: assume valor 1 caso a fábrica $i$ seja utilizada, e zero caso contrário, para toda fábrica $i=1,...,m$;

* $yy_j$: assuma valor 1 caso o centro $j$ seja utilizado, e zero caso contrário, para todo centro $j=1,...,n$;

* $x_{ij}$: quantidade de produto produzido em uma fábrica $i$ e transportado para um centro $j$, para todo $i=1,...m$ e para todo $j=1,...,n$.

* $y_{jk}$: quantidade de produto transportados do centro $j$ para o cliente $k$, para todo centro $j=1,...,n$ e para todo cliente $k=1,...,p$

A partir destas variáveis, tem-se:

$$
\text{minimize}~Z~=~\sum_{i=1}^{m}[f_{i}xx_{i} + \sum_{j=1}^{n}c_{ij}x_{ij}] + \sum_{j=1}^{n}[h_{j}yy_{j} + \sum_{k=1}^{p}t_{jk}y_{jk}] \tag{1}
$$

Sujeito a:

$$
\sum_{j=1}^{n}x_{ij} \leq g_{i}, \forall i=1,...,m \tag{2}
$$

$$
\sum_{k=1}^{p}y_{jk} \leq q_{j}, \forall j=1,...,n \tag{3}
$$

$$
\sum_{j=1}^{n}y_{jk}=d_{k}, \forall k=1,2,...,p \tag{4}
$$

$$
\sum_{i=1}^{m}x_{ij}-\sum_{k=1}^{p}y_{jk} = 0, \forall j=1,...,n \tag{5}
$$

$$
\sum_{j=1}^{n}x_{ij} \leq g_{i}xx_{i}, \forall i=1,...,m \tag{6}
$$

$$
\sum_{k=1}^{p}y_{jk} \leq q_{j}yy_{j}, \forall j=1,...,n \tag{7}
$$

$$
x_{ij} \in \mathbb{Z}^+, \forall i=1,...,m; \forall j=1,...,n \tag{8}
$$

$$
y_{jk} \in \mathbb{Z}^+, \forall j=1,...,n; \forall k=1,...,p \tag{9}
$$

$$
xx_i \in \{0, 1\}, \forall i=1,...,m \tag{10}
$$

$$
yy_j \in \{0, 1\}, \forall j=1,...,n \tag{11}
$$

Em que (1) minimiza a soma dos custos variáveis relacionados ao transporte dos itens das fábricas para os centros e dos centros para os clientes, incluindo o custo fixo de operação, caso haja, das fábricas e centros. As desigualdades em (2) garantem que a capacidade de produção de cada fábrica é repeitada, bem como (3) garantem que a capacidade de operação de cada um dos centros é respeitada. A igualdade em (4) garante que a demanda de cada cliente será atendida de forma exata. As igualdades em (5) são restrições de fluxo sobre cada centro. Ou seja, totos os produtos que chegam até um centro, devem ser entregues aos clientes, não havendo possibilidade de formação de estoque no centro. As desigualdades em (6) e (7) garantem que só haverá fluxo partindo de uma fábrico ou partindo de um centro se estes estiverem abertos. Por fim (8-11) são as restrições de domínio sobre as variáveis de decisão.

## Dados

Os dados para testar seu código são dados a seguir.

\

|m|n|p|
|---|---|---|
|2|3|3|

\

| |$1$ |$2$ |$3$|
|---|---|---|---|
|$d_k$| 80| 90 |40|

\

|$c_{ij}$| $1$| $2$| $3$|
|---|----|----|---|
|$1$ |10 | 8 |       20|
|$2$ | 20| 6 |       11|

\

|$t_{jk}$| $1$| $2$| $3$|
|---|---|---|---|
|$1$| 7  |  5 | 10|
|$2$| 5  |  4 |  3|       
|$3$| 10 |  8 |  6|

\

| | $1$ | $2$ |
|---|---|---|
|$g_i$ |120 |100 |
|$f_i$ |300 |250 |

\

| |$1$| $2$| $3$|
|---|---|---|---|
|$q_j$ |70 |90 |66|
|$h_j$ |60 |80 |100|